#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown Prepares storage, installs ComfyUI (v0.36.0+ for native YuE2 nodes), and creates model directories.

import os
import subprocess
from IPython.display import clear_output

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
!pip install -q uv
!cd {LOCAL_WORKSPACE} && uv pip install --system -q -r requirements.txt
!uv pip install --system -q xformers!=0.0.18 --extra-index-url https://download.pytorch.org/whl/cu121

# Create model folders
for folder in ["checkpoints", "audio_encoders", "input", "output"]:
    os.makedirs(os.path.join(LOCAL_WORKSPACE, "models", folder), exist_ok=True)

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. Download YuE2 Models
#@markdown Downloads the YuE2 3B checkpoint and (optionally) the SheetSage2 audio encoder for covers.

# --- What to download ---
DOWNLOAD_CHECKPOINT = True    #@param {type:"boolean"}
DOWNLOAD_SHEETSAGE2 = True   #@param {type:"boolean"}
#@markdown *SheetSage2 is only needed for the **Music Cover** workflow.*

import os
import subprocess
import urllib.parse

WORKSPACE = "/content/ComfyUI"

CHECKPOINT_URL = "https://huggingface.co/Comfy-Org/YuE2/resolve/main/checkpoints/yue2_3b_int8_convrot.safetensors"
SHEETSAGE2_URL = "https://huggingface.co/Comfy-Org/YuE2/resolve/main/audio_encoders/sheetsage2_bf16.safetensors"

DIRS = {
    "checkpoints":    os.path.join(WORKSPACE, "models/checkpoints"),
    "audio_encoders": os.path.join(WORKSPACE, "models/audio_encoders"),
}

print("⚡ [@CoinNoin] Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir, filename=None):
    if not url.strip(): return
    if not filename:
        parsed = urllib.parse.urlparse(url)
        filename = os.path.basename(parsed.path)
    dest = os.path.join(target_dir, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print(f"   ✅ Already Cached: {filename}")
        return
    print(f"   📥 Fetching: {filename}...")
    aria2_cmd = [
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-o", filename, url, "-d", target_dir
    ]
    subprocess.run(aria2_cmd, check=True)
    print(f"   ✅ Saved in {os.path.basename(target_dir)}")

if DOWNLOAD_CHECKPOINT:
    download_file(CHECKPOINT_URL, DIRS["checkpoints"])

if DOWNLOAD_SHEETSAGE2:
    download_file(SHEETSAGE2_URL, DIRS["audio_encoders"])
else:
    print("   ℹ️ SheetSage2 skipped (only needed for Music Cover).")

print("\n✅ [@CoinNoin] YuE2 assets secured!")

In [ ]:
#@title 3. YuE2 Text to Music
#@markdown Describe the music style, paste lyrics with section tags, and generate a complete song with vocals.

# --- Music Content ---
STYLE = "English, warm female vocal, contemporary pop, 96 BPM, piano, rounded electric bass, restrained drums, clear diction" #@param {type:"string"}
LYRICS = "[Verse] Morning light across the window City waking down below [Chorus] Run with me into the sunlight Leave the shadows far behind" #@param {type:"string"}

# --- Planning ---
ABC_PLANNING = True #@param {type:"boolean"}
#@markdown *If enabled, YuE2 first writes an editable ABC melody/chord plan before rendering audio.*
MODE = "full" #@param ["full", "melody"]
#@markdown *`full` = melody + chords; `melody` = melody only (recommended for covers).*
MAX_DURATION = 30 #@param {type:"slider", min:10, max:900, step:10}
#@markdown *Maximum song length in seconds. Generation can stop earlier if the model writes an end token.*

# --- Advanced ---
SEED = 0 #@param {type:"integer"}
#@markdown *0 = random seed.*
CFG_SCALE = 1.0 #@param {type:"number"}
#@markdown *1.0 disables guidance (matches ABC workflow). Use 1.01 for off-mode guidance.*
MAX_ABC_TOKENS = 8192 #@param {type:"integer"}

# --- Direct Download ---
AUTO_DOWNLOAD = False #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from IPython.display import display, Audio as IPAudio, clear_output

WORKSPACE = "/content/ComfyUI"
os.chdir(WORKSPACE)

UNET_FILENAME = "yue2_3b_int8_convrot.safetensors"

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py", "--port", "8188", "--lowvram"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
clear_output()

BRAND_PREFIX = "@CoinNoin_YuE2_T2M"

print(f"\033[94m➜ [@CoinNoin] Details | Mode: {MODE} | ABC: {ABC_PLANNING} | Max Dur: {MAX_DURATION}s | Seed: {SEED}\033[0m")

# Build workflow
prompt_workflow = {
    "ckpt": {"class_type": "CheckpointLoaderSimple",
             "inputs": {"ckpt_name": UNET_FILENAME}},

    "music": {"class_type": "YuE2GenerateMusic",
              "inputs": {
                  "clip": ["ckpt", 1],
                  "style": STYLE,
                  "lyrics": LYRICS,
                  "abc": "",          # will be replaced if ABC planning is on
                  "seed": SEED,
                  "mode": MODE,
                  "max_duration": float(MAX_DURATION),
                  "temperature": 1.0,
                  "top_p": 0.95,
                  "top_k": 100,
                  "repetition_penalty": 1.2,
                  "cfg_scale": float(CFG_SCALE)
              }},

    "latent": {"class_type": "EmptyYuE2LatentAudio",
               "inputs": {"seconds": ["music", 1], "batch_size": 1}},

    "zero": {"class_type": "ConditioningZeroOut",
             "inputs": {"conditioning": ["music", 0]}},

    "sampler": {"class_type": "KSampler",
                "inputs": {
                    "seed": SEED,
                    "steps": 32,
                    "cfg": 1.0,
                    "sampler_name": "dpm_2",
                    "scheduler": "sgm_uniform",
                    "denoise": 1.0,
                    "model": ["ckpt", 0],
                    "positive": ["music", 0],
                    "negative": ["zero", 0],
                    "latent_image": ["latent", 0]
                }},

    "decode": {"class_type": "VAEDecodeAudio",
               "inputs": {"samples": ["sampler", 0], "vae": ["ckpt", 2]}},

    "save": {"class_type": "SaveAudio",
             "inputs": {"audio": ["decode", 0], "filename_prefix": BRAND_PREFIX}},
}

# Optional ABC planning
if ABC_PLANNING:
    prompt_workflow["abc"] = {
        "class_type": "YuE2GenerateABC",
        "inputs": {
            "clip": ["ckpt", 1],
            "style": STYLE,
            "lyrics": LYRICS,
            "seed": SEED,
            "mode": MODE,
            "max_abc_tokens": MAX_ABC_TOKENS,
            "temperature": 0.7,
            "top_p": 0.9,
            "top_k": 30,
            "repetition_penalty": 1.005,
            "penalty_window": 100
        }
    }
    prompt_workflow["music"]["inputs"]["abc"] = ["abc", 0]
    print("✨ [@CoinNoin] ABC planning enabled")

# Submit
payload = json.dumps({"prompt": prompt_workflow}).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=payload)

print("\n📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Generating song (this can take several minutes on T4)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(2)

print("🎵 Decoding Final Song...")
generated_audio = []
for node_id, node_output in outputs.items():
    # SaveAudio output key can be "audio" or "files"
    for key in ("audio", "files"):
        if key in node_output:
            for item in node_output[key]:
                filename = item.get("filename") if isinstance(item, dict) else item
                if not filename:
                    continue
                audio_path = os.path.join(WORKSPACE, "output", filename)
                if os.path.exists(audio_path):
                    generated_audio.append(audio_path)
                    print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
                    display(IPAudio(filename=audio_path))

if AUTO_DOWNLOAD and generated_audio:
    print("\n📥 [@CoinNoin] Download Started...")
    for f in generated_audio:
        files.download(f)

In [ ]:
#@title 4. YuE2 Music Cover
#@markdown Upload a reference song. SheetSage2 transcribes its melody into ABC notation, then YuE2 re-sings it with your new style and lyrics.

# --- Upload ---
UPLOAD_REFERENCE_AUDIO = True #@param {type:"boolean"}
#@markdown *Upload a `.flac`, `.wav`, or `.mp3` reference song.*

# --- Music Content ---
STYLE = "English, soulful jazz-pop, expressive male vocal, relaxed 88 BPM, Rhodes piano, upright bass, brushed drums, warm saxophone, intimate late-night atmosphere" #@param {type:"string"}
LYRICS = "[Verse] Morning light across the window City waking down below [Chorus] Run with me into the sunlight Leave the shadows far behind" #@param {type:"string"}

# --- Cover Settings ---
COVER_MODE = "melody" #@param ["melody", "full"]
#@markdown *`melody` = transcription keeps only the melody (recommended for covers).*
MAX_DURATION = 30 #@param {type:"slider", min:10, max:900, step:10}

# --- Advanced ---
SEED = 0 #@param {type:"integer"}
CFG_SCALE = 1.0 #@param {type:"number"}

# --- Direct Download ---
AUTO_DOWNLOAD = False #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from IPython.display import display, Audio as IPAudio, clear_output

WORKSPACE = "/content/ComfyUI"
INPUT_DIR = os.path.join(WORKSPACE, "input")
os.makedirs(INPUT_DIR, exist_ok=True)
os.chdir(WORKSPACE)

UNET_FILENAME = "yue2_3b_int8_convrot.safetensors"
SHEETSAGE2_FILENAME = "sheetsage2_bf16.safetensors"

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py", "--port", "8188", "--lowvram"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
clear_output()

# Upload reference audio
reference_filename = "yue2_reference_song.flac"
if UPLOAD_REFERENCE_AUDIO:
    print("📤 [@CoinNoin] Please upload your REFERENCE song (flac/wav/mp3)...")
    uploaded = files.upload()
    if uploaded:
        original_name = list(uploaded.keys())[0]
        ext = os.path.splitext(original_name)[1].lower() or ".flac"
        reference_filename = f"yue2_reference_song{ext}"
        os.replace(original_name, os.path.join(INPUT_DIR, reference_filename))
        print(f"   ✅ Reference saved as: {reference_filename}")
        display(IPAudio(filename=os.path.join(INPUT_DIR, reference_filename)))
    else:
        print("   ❌ No file uploaded. Attempting to use cached reference.")
else:
    print("   ℹ️ Using previously uploaded reference audio.")

BRAND_PREFIX = "@CoinNoin_YuE2_Cover"

print(f"\n\033[94m➜ [@CoinNoin] Details | Cover Mode: {COVER_MODE} | Max Dur: {MAX_DURATION}s | Seed: {SEED}\033[0m")

# Build workflow
prompt_workflow = {
    "ckpt": {"class_type": "CheckpointLoaderSimple",
             "inputs": {"ckpt_name": UNET_FILENAME}},

    "load_audio": {"class_type": "LoadAudio",
                   "inputs": {"audio": reference_filename}},

    "encoder": {"class_type": "AudioEncoderLoader",
                "inputs": {"audio_encoder_name": SHEETSAGE2_FILENAME}},

    "transcribe": {"class_type": "SheetSage2AudioToABC",
                   "inputs": {
                       "audio_encoder": ["encoder", 0],
                       "audio": ["load_audio", 0],
                       "mode": COVER_MODE
                   }},

    "music": {"class_type": "YuE2GenerateMusic",
              "inputs": {
                  "clip": ["ckpt", 1],
                  "style": STYLE,
                  "lyrics": LYRICS,
                  "abc": ["transcribe", 0],
                  "seed": SEED,
                  "mode": COVER_MODE,
                  "max_duration": float(MAX_DURATION),
                  "temperature": 1.0,
                  "top_p": 0.95,
                  "top_k": 100,
                  "repetition_penalty": 1.2,
                  "cfg_scale": float(CFG_SCALE)
              }},

    "latent": {"class_type": "EmptyYuE2LatentAudio",
               "inputs": {"seconds": ["music", 1], "batch_size": 1}},

    "zero": {"class_type": "ConditioningZeroOut",
             "inputs": {"conditioning": ["music", 0]}},

    "sampler": {"class_type": "KSampler",
                "inputs": {
                    "seed": SEED,
                    "steps": 32,
                    "cfg": 1.0,
                    "sampler_name": "dpm_2",
                    "scheduler": "sgm_uniform",
                    "denoise": 1.0,
                    "model": ["ckpt", 0],
                    "positive": ["music", 0],
                    "negative": ["zero", 0],
                    "latent_image": ["latent", 0]
                }},

    "decode": {"class_type": "VAEDecodeAudio",
               "inputs": {"samples": ["sampler", 0], "vae": ["ckpt", 2]}},

    "save": {"class_type": "SaveAudio",
             "inputs": {"audio": ["decode", 0], "filename_prefix": BRAND_PREFIX}},
}

payload = json.dumps({"prompt": prompt_workflow}).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=payload)

print("\n📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Transcribing + generating cover (this can take several minutes on T4)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(2)

print("🎵 Decoding Cover Song...")
generated_audio = []
for node_id, node_output in outputs.items():
    for key in ("audio", "files"):
        if key in node_output:
            for item in node_output[key]:
                filename = item.get("filename") if isinstance(item, dict) else item
                if not filename:
                    continue
                audio_path = os.path.join(WORKSPACE, "output", filename)
                if os.path.exists(audio_path):
                    generated_audio.append(audio_path)
                    print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
                    display(IPAudio(filename=audio_path))

if AUTO_DOWNLOAD and generated_audio:
    print("\n📥 [@CoinNoin] Download Started...")
    for f in generated_audio:
        files.download(f)

In [ ]:
#@title 5. Notes & Tips
#@markdown Reference for prompting and parameters.

# --- Notes ---
#@markdown ### Style
#@markdown Describe **genre**, **vocal type**, **language**, **tempo**, and **instruments**. Keep musical direction in Style, not in Lyrics. Example:
#@markdown `English, warm female vocal, contemporary pop, 96 BPM, piano, rounded electric bass, restrained drums, clear diction`
#
#@markdown ### Lyrics
#@markdown Use section tags: `[Verse]`, `[Chorus]`, `[Bridge]`, `[Outro]`. Put only words intended to be sung in the Lyrics field.
#
#@markdown ### ABC Planning
#@markdown - **full**: melody + chords (recommended for text-to-music).
#@markdown - **melody**: melody only (recommended for covers).
#@markdown - Leave ABC empty (uncheck ABC_PLANNING) to generate directly from Style + Lyrics.
#
#@markdown ### Max Duration
#@markdown Ceiling in seconds (max 900). Generation budget is ~25 semantic tokens per second. The model stops early if it writes an end token. Long prompts shorten the song automatically.
#
#@markdown ### Guidance (cfg_scale)
#@markdown - `1.0` disables guidance (matches ABC workflow).
#@markdown - `1.01` matches the model's original off-mode guidance.
#
#@markdown ### T4 Notes
#@markdown YuE2 is a 3B model running two autoregressive passes + an ODE solver. Expect ~2–5 minutes per song on T4. The cover workflow adds SheetSage2 transcription (~30–60s). If OOM, reduce `max_duration` or close other notebooks.

print("📖 Notes loaded. Refer to the form above for YuE2 guidance.")